# Ernesto Investing AI — Notebook 03b: Clasificadores RNN (parte 2/2)

Entrena las arquitecturas **GRU** y **SimpleRNN** para clasificacion
binaria BUY/SELL. Continuacion del Notebook 03 (LSTM + BiLSTM).

**Requisito previo:** el Notebook 01 debe haberse ejecutado
al menos una vez. Este notebook es independiente del 03: puede
ejecutarse en otra sesion de Colab sin depender de sus variables.

## 1. Instalacion e importaciones

In [ ]:
!pip install "pymongo[srv]" --quiet

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from datetime import datetime, timezone

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Bidirectional, Dropout, Dense
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from pymongo import MongoClient
from google.colab import userdata

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Configuracion y conexion a MongoDB

In [ ]:
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]
VENTANA = 20      # dias de historia que ve el modelo para predecir el dia siguiente
EPOCAS = 80
BATCH_SIZE = 64

MONGO_URI = userdata.get("MONGO_URI")
MONGO_DB_NAME = "ernesto_investing_ai"

cliente = MongoClient(MONGO_URI)
db = cliente[MONGO_DB_NAME]

col_precios = db["precios_ohlcv"]
col_predicciones = db["predicciones"]
col_metricas = db["metricas_modelos"]

print("Conexion exitosa. Documentos en precios_ohlcv:", col_precios.count_documents({}))

## 3. Carga de datos y construccion de secuencias

Cada secuencia de `VENTANA` dias predice si el precio de cierre del
dia **siguiente** a la secuencia sube (1) o baja (0) respecto al
ultimo dia de la secuencia. La ultima secuencia disponible (sin
target, porque "mañana" todavia no paso) se guarda aparte para
generar la senal actual del sistema.

In [ ]:
COLUMNAS_FEATURES = [
    "close", "sma_20", "sma_50", "ema_12", "ema_26", "rsi_14",
    "macd", "macd_signal", "macd_hist", "bb_upper", "bb_lower",
]


def cargar_precios(ticker: str) -> pd.DataFrame:
    """Lee todos los documentos de un ticker desde precios_ohlcv, ordenados por fecha."""
    cursor = col_precios.find({"ticker": ticker}).sort("fecha", 1)
    filas = []
    for doc in cursor:
        fila = {"fecha": doc["fecha"], "close": doc["close"]}
        fila.update(doc["indicadores"])
        filas.append(fila)
    return pd.DataFrame(filas).dropna(subset=COLUMNAS_FEATURES).reset_index(drop=True)


def construir_secuencias(df: pd.DataFrame, ventana: int = VENTANA):
    """
    Devuelve (X, y, secuencia_actual):
      X                -> array (muestras, ventana, n_features) para entrenar
      y                -> 1 si el cierre del dia i supera al del dia i-1, 0 si no
      secuencia_actual -> ultima ventana de dias, para predecir la senal de "manana"
    """
    valores = df[COLUMNAS_FEATURES].values
    cierres = df["close"].values
    n = len(df)

    X, y = [], []
    for i in range(ventana, n):
        X.append(valores[i - ventana:i])
        y.append(1 if cierres[i] > cierres[i - 1] else 0)

    secuencia_actual = valores[n - ventana:n]
    return np.array(X), np.array(y), secuencia_actual

## 4. Escalado y particion temporal

`MinMaxScaler` se ajusta solo con datos de entrenamiento (aplanando
la ventana temporal) y se reutiliza para transformar test y la
secuencia actual, evitando fuga de informacion del futuro.

In [ ]:
def escalar_y_dividir(X: np.ndarray, y: np.ndarray, secuencia_actual: np.ndarray):
    """Divide 80/20 en orden temporal, ajusta el scaler solo con train."""
    n_train = int(len(X) * 0.8)
    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]

    n_muestras, ventana, n_features = X_train.shape

    scaler = MinMaxScaler()
    scaler.fit(X_train.reshape(-1, n_features))

    def transformar(arreglo):
        forma_original = arreglo.shape
        plano = arreglo.reshape(-1, n_features)
        return scaler.transform(plano).reshape(forma_original)

    X_train_esc = transformar(X_train)
    X_test_esc = transformar(X_test)
    secuencia_actual_esc = scaler.transform(secuencia_actual).reshape(1, ventana, n_features)

    return X_train_esc, X_test_esc, y_train, y_test, secuencia_actual_esc

## 5. Entrenamiento generico y guardado en MongoDB

Esta funcion es la misma para las arquitecturas de este notebook:
solo cambia la funcion que construye el modelo. Guarda la senal
actual en `predicciones` y las metricas + curva de entrenamiento
en `metricas_modelos` (la curva de loss/accuracy por epoca queda
disponible para el modulo de "Curvas de entrenamiento" del frontend).

In [ ]:
def entrenar_arquitectura(nombre_modelo: str, construir_modelo_fn, ventana: int = VENTANA):
    resumen = {}

    for ticker in TICKERS:
        print(f"[{nombre_modelo}] Procesando {ticker}...")
        try:
            df = cargar_precios(ticker)
            if len(df) < ventana + 40:
                print(f"  {ticker}: insuficientes datos, se omite")
                continue

            X, y, secuencia_actual = construir_secuencias(df, ventana)
            X_train, X_test, y_train, y_test, secuencia_actual_esc = escalar_y_dividir(X, y, secuencia_actual)

            modelo = construir_modelo_fn(input_shape=(ventana, X.shape[2]))
            historial = modelo.fit(
                X_train, y_train,
                epochs=EPOCAS, batch_size=BATCH_SIZE,
                validation_split=0.15, verbose=0,
            )

            probabilidades_test = modelo.predict(X_test, verbose=0).flatten()
            predicciones_test = (probabilidades_test > 0.5).astype(int)

            metricas = {
                "accuracy": float(accuracy_score(y_test, predicciones_test)),
                "precision": float(precision_score(y_test, predicciones_test, zero_division=0)),
                "recall": float(recall_score(y_test, predicciones_test, zero_division=0)),
                "f1": float(f1_score(y_test, predicciones_test, zero_division=0)),
                "rmse": None, "mae": None, "r2": None,
            }
            matriz = confusion_matrix(y_test, predicciones_test).tolist()

            probabilidad_actual = float(modelo.predict(secuencia_actual_esc, verbose=0)[0][0])
            senal_actual = "BUY" if probabilidad_actual > 0.5 else "SELL"

            historial_entrenamiento = [
                {
                    "epoca": i + 1,
                    "loss": float(historial.history["loss"][i]),
                    "accuracy": float(historial.history["accuracy"][i]),
                    "val_loss": float(historial.history["val_loss"][i]),
                    "val_accuracy": float(historial.history["val_accuracy"][i]),
                }
                for i in range(len(historial.history["loss"]))
            ]

            ahora = datetime.now(timezone.utc)

            col_predicciones.insert_one({
                "ticker": ticker, "modelo": nombre_modelo, "tipo": "clasificacion",
                "fecha_generacion": ahora, "senal": senal_actual,
                "probabilidad": round(probabilidad_actual if senal_actual == "BUY" else 1 - probabilidad_actual, 4),
                "precio_predicho": None, "horizonte_dias": None,
                "banda_confianza": {"inferior": None, "superior": None},
            })

            col_metricas.update_one(
                {"ticker": ticker, "modelo": nombre_modelo},
                {"$set": {
                    "ticker": ticker, "modelo": nombre_modelo,
                    "fecha_entrenamiento": ahora, "metricas": metricas,
                    "hiperparametros": {"ventana": ventana, "epocas": EPOCAS, "batch_size": BATCH_SIZE},
                    "matriz_confusion": matriz,
                    "historial_entrenamiento": historial_entrenamiento,
                }},
                upsert=True,
            )

            resumen[ticker] = {"senal": senal_actual, "accuracy": round(metricas["accuracy"], 3)}
            print(f"  {ticker}: senal={senal_actual}, accuracy={metricas['accuracy']:.3f}")

        except Exception as error:
            print(f"  ERROR con {ticker}: {error}")
            resumen[ticker] = f"ERROR: {error}"

    return resumen

## 6 — GRU

Sin capas de Dropout: diseno intencional para comparar contra LSTM/BiLSTM.

In [ ]:
def construir_modelo_gru(input_shape):
    """GRU(280, return_sequences=True) -> GRU(140) -> Dense(1, sigmoid). Sin Dropout (diseno intencional)."""
    modelo = Sequential([
        GRU(280, return_sequences=True, input_shape=input_shape),
        GRU(140),
        Dense(1, activation="sigmoid"),
    ])
    modelo.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
    return modelo

In [ ]:
resumen_gru = entrenar_arquitectura("gru", construir_modelo_gru)
print()
print("Resumen GRU:", resumen_gru)

## 7 — SimpleRNN

Baseline simple: sirve para verificar si las arquitecturas con compuertas realmente aportan.

In [ ]:
def construir_modelo_simplernn(input_shape):
    """SimpleRNN(180, return_sequences=True) -> SimpleRNN(90) -> Dense(1, sigmoid). Baseline de comparacion."""
    modelo = Sequential([
        SimpleRNN(180, return_sequences=True, input_shape=input_shape),
        SimpleRNN(90),
        Dense(1, activation="sigmoid"),
    ])
    modelo.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
    return modelo

In [ ]:
resumen_simplernn = entrenar_arquitectura("simplernn", construir_modelo_simplernn)
print()
print("Resumen SimpleRNN:", resumen_simplernn)

## Verificacion final

In [ ]:
for ticker in TICKERS:
    ultima_gru = col_predicciones.find_one({"ticker": ticker, "modelo": "gru"}, sort=[("fecha_generacion", -1)])
    if ultima_gru:
        print(f"{ticker} [GRU]: senal={ultima_gru['senal']}, probabilidad={ultima_gru['probabilidad']}")
    ultima_simplernn = col_predicciones.find_one({"ticker": ticker, "modelo": "simplernn"}, sort=[("fecha_generacion", -1)])
    if ultima_simplernn:
        print(f"{ticker} [SimpleRNN]: senal={ultima_simplernn['senal']}, probabilidad={ultima_simplernn['probabilidad']}")